# W2D3 — Clean and Engineer — Lab

**Week 2 · Day 3 · Data Engineering** · Lab

D1 found the problems and D2 found the patterns. Today you fix the first and build on the second.
You will de-duplicate, collapse fourteen city spellings into five, parse four date formats into one
real date, handle a column that is 29% missing *for a reason*, and turn the patterns D2 surfaced
into engineered columns. Then you will do the most important thing in the week: decide which
columns you are **allowed** to use, and prove that one of them is leakage. You leave with
`sales_model_ready.parquet` and `sales_feature_spec.json` — the two files D5 trains on.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٢ اليوم ٣ — التنظيف والهندسة

**الأسبوع ٢ · اليوم ٣ · هندسة البيانات** · معمل عملي

وجد اليوم الأول المشكلات، ووجد اليوم الثاني الأنماط. واليوم تصلح الأولى وتبني على الثانية. ستحذف
المكرّرات، وتجمع أربع عشرة تهجئة للمدن في خمس، وتحوّل أربع صيغ تاريخ إلى تاريخ حقيقي واحد، وتعالج
عمودًا ناقصًا بنسبة ٢٩٪ **لسبب**، وتحوّل الأنماط التي أظهرها اليوم الثاني إلى أعمدة مهندسة. ثم
تفعل أهم شيء في الأسبوع: تقرّر أي الأعمدة **يُسمح** لك باستخدامها، وتثبت أن أحدها تسريب. وستخرج
بملفَّي `sales_model_ready.parquet` و`sales_feature_spec.json` اللذين يتدرّب عليهما اليوم الخامس.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Remove duplicate rows and show that removing them did not distort the target.
- Normalise inconsistent category spellings and verify the collapse was complete.
- Parse a text column holding several date formats, and engineer features from the result.
- Choose an imputation strategy that fits *why* a value is missing, and keep the missingness itself as a feature.
- State the moment a prediction would be made, and use it to decide whether a column is allowed.
- Demonstrate that a column is leakage rather than asserting it, and exclude it without deleting it.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- حذف الصفوف المكرّرة وإظهار أن حذفها لم يشوّه الهدف.
- توحيد تهجئات الفئات غير المتّسقة والتحقّق من اكتمال التجميع.
- تحليل عمود نصّي يحمل عدة صيغ تاريخ، وهندسة خصائص من الناتج.
- اختيار استراتيجية تعويض تناسب **سبب** غياب القيمة، مع الإبقاء على الغياب نفسه كخاصية.
- تحديد اللحظة التي يُتّخذ فيها التنبّؤ، واستخدامها لتقرير ما إذا كان العمود مسموحًا.
- إثبات أن عمودًا ما تسريب بدل الادّعاء بذلك، واستبعاده دون حذفه.

</div>

## About the data

**Dataset:** `messy_sales` — built for this course · CC0 · 5,000 rows

The same orders file for the third day. One row is one order; the target is `returned`.

**The prediction moment.** Everything today hangs on one decision, so make it explicitly: the shop
wants to flag at-risk parcels **as they leave the warehouse**. So at prediction time you know the
order details, the shipping dates and the revenue that was recognised. You do **not** know anything
that only exists once the customer has reacted.

**Watch out:** `refund_amount` correlates with `returned` far more strongly than any honest column.
That is not a gift. A refund is only raised after a return is accepted, which is strictly after the
moment above — so it cannot be a feature, however good it looks.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `messy_sales` — أُعدّت لهذه الدورة · رخصة CC0 · ٥٠٠٠ صف

ملف الطلبات نفسه لليوم الثالث. الصف الواحد طلب واحد، والهدف هو `returned`.

**لحظة التنبّؤ.** يتوقّف كل شيء اليوم على قرار واحد، فاتّخذه صراحةً: يريد المتجر تمييز الطرود
المعرّضة للخطر **عند مغادرتها المستودع**. فأنت تعرف وقت التنبّؤ تفاصيل الطلب وتواريخ الشحن
والإيراد المعترف به، ولا تعرف أي شيء لا يوجد إلا بعد أن يتفاعل العميل.

**انتبه:** يرتبط `refund_amount` بـ`returned` أقوى بكثير من أي عمود نزيه. وهذه ليست هدية. فالمبلغ
المسترد لا يُفتح إلا بعد قبول الإرجاع، أي بعد اللحظة أعلاه قطعًا — فلا يصلح خاصيةً مهما بدا جيدًا.

</div>

## Setup

Run the cell below first. It installs anything missing, fixes the random seed, and finds the
dataset — whether you are on your own machine or on Google Colab.

<div dir="rtl" align="right">

## الإعداد

شغّل الخلية التالية أولًا. تُثبّت ما ينقص، وتُثبّت البذرة العشوائية، وتجد ملف البيانات — سواء كنت
على جهازك أو على Google Colab.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("scikit-learn")                   # one small model, used only as a measuring stick
seed_everything(42)

import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

DATA = get_dataset("messy_sales")
print(describe_dataset("messy_sales"))
print("\n", versions(), "| device:", device())

## Section 1 — Warm-up  (≈25 min)

Guided. The code works. The first two fixes are done for you so you can see the shape of a cleaning
step: measure before, change, measure after. Every fix in Section 2 follows that same shape.

<div dir="rtl" align="right">

## القسم الأول — التهيئة (نحو ٢٥ دقيقة)

قسم موجَّه. الشيفرة تعمل. أول إصلاحين مُنجزان لك لترى شكل خطوة التنظيف: قِس قبل، غيّر، قِس بعد.
وكل إصلاح في القسم الثاني يتبع الشكل نفسه.

</div>

In [ ]:
# Working code. Nothing to fill in here.
raw = pd.read_csv(DATA)

# `clean` is the frame we build up through the lab. `raw` stays untouched so every
# step can be compared against the original.
clean = raw.copy()

print(f"loaded {len(raw):,} rows x {raw.shape[1]} columns")
print(f"returned rate: {raw['returned'].mean():.4f}")
raw.head(3)

### Task 1.1 — Fix 1: the duplicates (worked example)

Measure, change, measure. Note that the check is not "did the rows go away" — it is "did removing
them move the thing I care about". Run it.

<div dir="rtl" align="right">

### المهمة ١٫١ — الإصلاح الأول: المكرّرات (مثال محلول)

قِس، غيّر، قِس. ولاحظ أن الفحص ليس «هل اختفت الصفوف» بل «هل حرّك حذفها ما يهمّني». شغّله.

</div>

In [ ]:
# Worked example — read it, run it, then copy this shape in Section 2.
before = {"rows": len(clean), "returned": clean["returned"].mean()}

n_duplicates = int(clean.duplicated().sum())
clean = clean.drop_duplicates().reset_index(drop=True)

after = {"rows": len(clean), "returned": clean["returned"].mean()}

print(f"removed {n_duplicates} duplicate rows: {before['rows']:,} -> {after['rows']:,}")
print(f"returned rate: {before['returned']:.4f} -> {after['returned']:.4f} "
      f"({after['returned'] - before['returned']:+.4f})")
print("\nThe target barely moved, so the duplicates were not a biased subset —"
      "\nremoving them is safe rather than merely convenient.")

### Task 1.2 — Fix 2: the city spellings (worked example)

`city` has fourteen spellings of five cities: stray whitespace and inconsistent capitalisation.
Stripping and title-casing collapses every variant onto its canonical form.

Change the chained operations below — try removing `.str.strip()` — and see the count stop
collapsing to five.

<div dir="rtl" align="right">

### المهمة ١٫٢ — الإصلاح الثاني: تهجئات المدن (مثال محلول)

يحمل `city` أربع عشرة تهجئة لخمس مدن: مسافات زائدة وحالة أحرف غير متّسقة. وإزالة المسافات وتوحيد
حالة الأحرف يجمع كل تنويع على صورته المعيارية.

غيّر العمليات المتسلسلة أدناه — جرّب حذف `.str.strip()` — وشاهد العدد يتوقّف عن الانخفاض إلى خمسة.

</div>

In [ ]:
# Try removing .str.strip() or .str.title() and re-run to see what stops working.
spellings_before = clean["city"].nunique()
clean["city"] = clean["city"].str.strip().str.title()

print(f"city: {spellings_before} spellings -> {clean['city'].nunique()} cities")
print(sorted(clean["city"].unique()))

## Section 2 — Core  (≈60 min)

This is the lab. Each task has a goal; you write the code. Keep building up the same `clean` frame.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي (نحو ٦٠ دقيقة)

هذا هو صلب المعمل. لكل مهمة هدف، وأنت من يكتب الشيفرة. واصل البناء على الإطار `clean` نفسه.

</div>

### Task 2.1 — Parse the dates, then build features from them

`order_date` is text in four formats and `ship_date` is text in one. Parse both into real
datetimes, then engineer the columns that the dates make possible:

- how many days the order took to ship,
- the month it was placed,
- the day of the week it was placed.

Verify the parse rather than trusting it: nothing may fail to parse, nothing may land in the
future, and nothing may ship before it was ordered.

<div dir="rtl" align="right">

### المهمة ٢٫١ — حلّل التواريخ ثم ابنِ منها خصائص

`order_date` نص بأربع صيغ و`ship_date` نص بصيغة واحدة. حلّلهما إلى تواريخ حقيقية، ثم اهندس
الأعمدة التي تتيحها التواريخ:

- كم يومًا استغرق شحن الطلب،
- الشهر الذي قُدّم فيه،
- يوم الأسبوع الذي قُدّم فيه.

وتحقّق من التحليل بدل الثقة به: يجب ألا يفشل شيء في التحليل، وألا يقع شيء في المستقبل، وألا يُشحن
شيء قبل طلبه.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) One of the two columns holds several written formats. pandas has an argument
#    that lets it handle a mixed column in one call — find it.
# 2) Subtracting two datetime columns gives a duration; you want it in whole days.
# 3) A parsed datetime column exposes its parts through an accessor — month and
#    day-of-week come from there.
# 4) Count three things to prove the parse worked: failures, future dates, and
#    orders that shipped before they were placed.
# Search: "pandas to_datetime format mixed dt accessor"
# https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
#
# ١) يحمل أحد العمودين عدة صيغ مكتوبة. ولدى pandas وسيط يتيح له التعامل مع عمود
#    مختلط في استدعاء واحد — جِده.
# ٢) طرح عمودَي تاريخ يعطي مدة؛ وأنت تريدها بالأيام الكاملة.
# ٣) يكشف عمود التاريخ المحلَّل أجزاءه عبر واصل — ومنه يأتي الشهر
#    ويوم الأسبوع.
# ٤) عُدّ ثلاثة أشياء لإثبات نجاح التحليل: الإخفاقات، والتواريخ المستقبلية،
#    والطلبات التي شُحنت قبل تقديمها.
# ابحث عن: "pandas to_datetime format mixed dt accessor"
# ────────────────────────────────────────────────────────────────────

# TODO: Parse both date columns into real datetimes.
# مهمة: حلّل عمودَي التاريخ إلى تواريخ حقيقية.
# TODO: Engineer ship_delay_days, order_month and order_dow from the parsed dates.
# مهمة: اهندس ship_delay_days وorder_month وorder_dow من التواريخ المحلَّلة.
# TODO: Count the three ways the parse could be wrong: unparsed, future, shipped-before-ordered.
# مهمة: عُدّ الطرق الثلاث لخطأ التحليل: غير محلَّل، مستقبلي، شُحن قبل الطلب.
print(f"unparsed: {n_unparsed} | future: {n_future} | shipped before ordered: {n_before_order}")
print(f"\nship_delay_days: {clean['ship_delay_days'].min()}-{clean['ship_delay_days'].max()} days")
print(clean[["order_date", "order_dt", "ship_delay_days", "order_month", "order_dow"]].head())

### Task 2.2 — Impute the discount, and keep the fact that it was missing

`discount_pct` is missing for about 29% of rows, and D1 showed the reason is not random: **every
phone order is missing it**, because that channel never captured the field.

That changes what you are allowed to do. Filling the gap with the median is fine as a number, but
if you stop there you have thrown away a real signal — "this order came through a channel that
does not record discounts" is information, and after imputation it is invisible.

So do both: record the missingness in its own indicator column **first**, then fill the values.
Order matters.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — عوّض الخصم، واحتفظ بحقيقة أنه كان مفقودًا

`discount_pct` مفقود في نحو ٢٩٪ من الصفوف، وقد أظهر اليوم الأول أن السبب ليس عشوائيًّا: **كل طلب
هاتفي يفتقده**، لأن تلك القناة لم تسجّل الحقل قط.

وهذا يغيّر ما يُسمح لك بفعله. فملء الفجوة بالوسيط مقبول كرقم، لكنك إن توقّفت عند ذلك تكون قد
أهدرت إشارة حقيقية — فعبارة «جاء هذا الطلب عبر قناة لا تسجّل الخصومات» معلومة، وتصبح غير مرئية
بعد التعويض.

فافعل الأمرين: سجّل الغياب في عمود مؤشّر خاص به **أولًا**، ثم املأ القيم. والترتيب مهم.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Create a 0/1 indicator column that records where the value was missing.
#    Do this BEFORE you fill anything, or the indicator will be all zeros.
# 2) Fill the remaining gaps with a statistic that is not dragged around by
#    outliers — the mean is not that statistic.
# 3) Confirm afterwards that nothing is still missing and the indicator kept its share.
# Search: "pandas fillna median missing indicator"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html
#
# ١) أنشئ عمود مؤشّر من صفر وواحد يسجّل مواضع غياب القيمة. افعل ذلك **قبل** أي
#    ملء، وإلا كان المؤشّر أصفارًا كلها.
# ٢) املأ الفجوات المتبقّية بإحصاء لا تجرّه القيم الشاذّة — والمتوسّط
#    ليس ذلك الإحصاء.
# ٣) تأكّد بعدها أن لا شيء ما زال مفقودًا وأن المؤشّر حافظ على حصته.
# ابحث عن: "pandas fillna median missing indicator"
# ────────────────────────────────────────────────────────────────────

missing_rate_before = clean["discount_pct"].isna().mean()
# TODO: Record where discount_pct was missing in an indicator column, then fill the gaps with a statistic robust to outliers.
# مهمة: سجّل مواضع غياب discount_pct في عمود مؤشّر، ثم املأ الفجوات بإحصاء مقاوم للقيم الشاذّة.
print(f"discount_pct was {missing_rate_before:.1%} missing, filled with "
      f"median {discount_median:.3f}")
print(f"still missing: {clean['discount_pct'].isna().sum()}")
print(f"indicator is 1 for {clean['discount_missing'].mean():.1%} of rows")
print("\nreturn rate by the indicator — if these differ, the missingness itself "
      "carries signal:")
print(clean.groupby("discount_missing")["returned"].agg(
    return_rate="mean", orders="size").to_string(
    formatters={"return_rate": "{:.1%}".format}))

### Task 2.3 — Engineer the features D2 pointed at

D2 found that the return rate moves with discount depth, shipping delay and order size. Build the
columns that let a model use those:

- `gross` — what the order was worth before the discount,
- `is_bulk` — whether the order is unusually large, using a threshold you can justify,
- `is_deep_discount` — whether the discount is in the top band D2 flagged.

For each, print the return rate on each side of the split. A feature that shows the same rate on
both sides is not a feature; find out now rather than in D5.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — اهندس الخصائص التي أشار إليها اليوم الثاني

وجد اليوم الثاني أن نسبة الإرجاع تتحرّك مع عمق الخصم وتأخّر الشحن وحجم الطلب. ابنِ الأعمدة التي
تتيح للنموذج استخدام ذلك:

- `gross` — قيمة الطلب قبل الخصم،
- `is_bulk` — هل الطلب كبير على نحو غير معتاد، بعتبة تستطيع تبريرها،
- `is_deep_discount` — هل الخصم في الشريحة العليا التي أبرزها اليوم الثاني.

واطبع لكل منها نسبة الإرجاع على جانبَي التقسيم. فالخاصية التي تُظهر النسبة نفسها على الجانبين ليست
خاصية؛ واكتشف ذلك الآن لا في اليوم الخامس.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) `gross` is the straightforward product of the two order-size columns.
# 2) For the two flags, pick a cut point from the data rather than inventing one —
#    a high quantile is defensible and you can say why you chose it.
# 3) Store the flags as 0/1 integers, not booleans, so they survive a round trip
#    through parquet unchanged.
# 4) For each new flag, group by it and print the return rate and the group size.
# Search: "pandas quantile astype int groupby mean"
#
# ١) `gross` هو حاصل ضرب عمودَي حجم الطلب مباشرة.
# ٢) اختر نقطة القطع للعلامتين من البيانات بدل اختراعها — فالشريحة المئوية
#    العليا مبرَّرة وتستطيع بيان سبب اختيارك.
# ٣) خزّن العلامات كأعداد صحيحة صفر وواحد لا كقيم منطقية، لتبقى كما هي بعد
#    رحلة ذهاب وعودة عبر parquet.
# ٤) جمّع حسب كل علامة جديدة واطبع نسبة الإرجاع وحجم المجموعة.
# ابحث عن: "pandas quantile astype int groupby mean"
# ────────────────────────────────────────────────────────────────────

# TODO: Build gross, is_bulk and is_deep_discount, choosing the thresholds from the data.
# مهمة: ابنِ gross وis_bulk وis_deep_discount، واختر العتبات من البيانات.
ENGINEERED = ["is_bulk", "is_deep_discount", "discount_missing"]
for col in ENGINEERED:
    table = clean.groupby(col)["returned"].agg(
        return_rate="mean", orders="size")
    print(f"--- {col} ---")
    print(table.to_string(formatters={"return_rate": "{:.1%}".format}), "\n")

### Task 2.4 — Prove `refund_amount` is leakage

You have suspected this column since D2. Now demonstrate it, because "it correlates too well" is a
hunch and a measurement is an argument.

Fit the same simple model twice on the same split — once on the honest columns, once on the honest
columns **plus** `refund_amount` — and compare the ROC AUC.

The gap you are about to see is not a good result. It is the size of the lie a leaked column tells
you about your own model.

The model here is a deliberately small one used as a measuring stick — D5 builds the real thing, so
do not read the honest number as "the best we can do".

<div dir="rtl" align="right">

### المهمة ٢٫٤ — أثبت أن `refund_amount` تسريب

تشكّ في هذا العمود منذ اليوم الثاني. والآن برهن على ذلك، فـ«ارتباطه جيد أكثر من اللازم» حدس،
والقياس حجّة.

درّب النموذج البسيط نفسه مرتين على التقسيم نفسه — مرة على الأعمدة النزيهة، ومرة على الأعمدة
النزيهة **مضافًا إليها** `refund_amount` — وقارن المساحة تحت المنحنى.

والفجوة التي ستراها ليست نتيجة جيدة، بل هي حجم الكذبة التي يقولها لك عمود مُسرَّب عن نموذجك.

والنموذج المستخدم هنا صغير عمدًا ويُستعمل كمسطرة قياس — واليوم الخامس يبني النموذج الحقيقي، فلا
تقرأ الرقم النزيه على أنه «أقصى ما نستطيع».

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Split once, stratified on the target, and reuse that split for both fits —
#    two different splits would confound the comparison you are trying to make.
# 2) Use a small tree-based classifier; it needs no scaling and handles the
#    zero-inflated refund column without fuss.
# 3) Score with ROC AUC on predicted probabilities, not with accuracy — the classes
#    are imbalanced and accuracy would hide the difference.
# 4) Fit twice: honest columns, then the same columns plus the suspect one.
# Search: "sklearn roc_auc_score predict_proba train_test_split stratify"
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html
#
# ١) قسّم مرة واحدة بتقسيم طبقي على الهدف، وأعد استخدام التقسيم نفسه للتدريبين —
#    فتقسيمان مختلفان يفسدان المقارنة التي تحاول إجراءها.
# ٢) استخدم مصنّفًا شجريًّا صغيرًا؛ فهو لا يحتاج معايرة ويتعامل مع عمود الاسترداد
#    المليء بالأصفار دون عناء.
# ٣) قِس بالمساحة تحت المنحنى على الاحتمالات لا بالدقّة — فالفئات غير متوازنة
#    وستخفي الدقّةُ الفرق.
# ٤) درّب مرتين: الأعمدة النزيهة، ثم الأعمدة نفسها مضافًا إليها المشبوه.
# ابحث عن: "sklearn roc_auc_score predict_proba train_test_split stratify"
# ────────────────────────────────────────────────────────────────────

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
HONEST_NUMERIC = ["quantity", "unit_price", "discount_pct", "ship_delay_days",
                  "order_month", "order_dow", "gross", "is_bulk",
                  "is_deep_discount", "discount_missing", "revenue"]
LEAK = "refund_amount"
# TODO: Make one stratified split, then score the same model with and without the leaking column, using ROC AUC on the held-out set.
# مهمة: أنشئ تقسيمًا طبقيًّا واحدًا، ثم قِس النموذج نفسه بوجود العمود المُسرِّب وبدونه، بالمساحة تحت المنحنى على مجموعة الاختبار.
print(f"honest columns only:      AUC = {honest_auc:.4f}")
print(f"with {LEAK}:   AUC = {leaked_auc:.4f}")
print(f"the lie:                        {leaked_auc - honest_auc:+.4f}")

**Why this is leakage, in three parts.** Write your version of each:

1. **When does the value exist?** A refund is raised only after a return has been accepted — strictly
   after the parcel leaves the warehouse, which is the moment we said we would predict at.
2. **Could it be derived from anything we legitimately have?** *(your answer)*
3. **What would happen in production?** *(your answer — think about what `refund_amount` would
   contain for an order you are predicting on right now, today, before anyone has returned anything)*

<div dir="rtl" align="right">

**لماذا هذا تسريب، في ثلاثة أجزاء.** اكتب صيغتك لكل جزء:

١. **متى توجد القيمة؟** لا يُفتح المبلغ المسترد إلا بعد قبول الإرجاع — أي بعد مغادرة الطرد
   المستودع قطعًا، وهي اللحظة التي قلنا إننا سنتنبّأ عندها.
٢. **هل يمكن اشتقاقه من أي شيء نملكه بوجه مشروع؟** *(إجابتك)*
٣. **ماذا سيحدث في الإنتاج؟** *(إجابتك — فكّر فيما سيحتويه `refund_amount` لطلب تتنبّأ عليه الآن،
   اليوم، قبل أن يُرجع أحد أي شيء)*

</div>

### Task 2.5 — Decide the feature set

Now write the decision down in a form a machine can read. Build three lists:

- **features** — the columns a model is allowed to use at the prediction moment,
- **target** — `returned`,
- **excluded** — every column you are leaving out, each with the reason.

Two columns are excluded for *different* reasons, and the distinction matters: `refund_amount` is
leakage, while `commission_paid` is simply not known yet at dispatch. Identifiers and the raw text
versions of columns you already parsed are excluded for a third, duller reason.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — قرّر مجموعة الخصائص

دوّن القرار الآن بصيغة تقرؤها الآلة. ابنِ ثلاث قوائم:

- **features** — الأعمدة المسموح للنموذج باستخدامها في لحظة التنبّؤ،
- **target** — `returned`،
- **excluded** — كل عمود تستبعده مع سبب استبعاده.

ويُستبعد عمودان لسببين **مختلفين**، والتمييز مهم: فـ`refund_amount` تسريب، أما `commission_paid`
فغير معروف بعد عند الشحن فحسب. وتُستبعد المعرّفات والنسخ النصّية الخام للأعمدة التي حلّلتها بالفعل
لسبب ثالث أقل إثارة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Start from the categorical columns and the honest numeric list you already have.
# 2) Build a dict mapping every excluded column to a one-line reason. Writing the
#    reason down is the point of the task — a bare list rots the moment someone
#    asks "why is that not in there?"
# 3) Check that features, target and excluded together account for every column in
#    the frame, with no column in two lists.
# Search: "python set difference symmetric_difference"
#
# ١) ابدأ من الأعمدة الفئوية وقائمة الأعمدة الرقمية النزيهة التي لديك.
# ٢) ابنِ قاموسًا يربط كل عمود مستبعد بسبب من سطر واحد. وتدوين السبب هو جوهر
#    المهمة — فالقائمة المجرّدة تتعفّن لحظة أن يسأل أحدهم
#    «ولماذا هذا ليس فيها؟»
# ٣) تحقّق من أن features وtarget وexcluded تغطّي معًا كل عمود في الإطار، وألا
#    يكون عمود في قائمتين.
# ابحث عن: "python set difference symmetric_difference"
# ────────────────────────────────────────────────────────────────────

CATEGORICAL = ["city", "channel", "customer_tier"]
# TODO: Define FEATURES, TARGET and EXCLUDED (column -> reason) so that every column in `clean` is accounted for exactly once.
# مهمة: عرّف FEATURES وTARGET وEXCLUDED (عمود ← سبب) بحيث يُحسب كل عمود في `clean` مرة واحدة بالضبط.
accounted = set(FEATURES) | {TARGET} | set(EXCLUDED)
unaccounted = set(clean.columns) - accounted
overlap = set(FEATURES) & set(EXCLUDED)
print(f"{len(FEATURES)} features, 1 target, {len(EXCLUDED)} excluded")
print(f"columns not accounted for: {sorted(unaccounted) or 'none'}")
print(f"columns in two lists:      {sorted(overlap) or 'none'}\n")
for col, reason in EXCLUDED.items():
    print(f"  {col:18s} {reason}")

### Task 2.6 — Save the handover

D5 needs two things: the table, and the decision about the table. Save both.

The parquet keeps `refund_amount` in it **on purpose** — D5 reproduces the leakage demonstration,
and it cannot do that with a column that was deleted here. The JSON is what stops that from being
dangerous: it records which columns are features, so D5 has to opt in to the leak deliberately
rather than by accident.

That pairing — the data plus the contract describing it — is the habit worth taking to your
capstone.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — احفظ التسليم

يحتاج اليوم الخامس شيئين: الجدول، والقرار المتعلّق بالجدول. احفظ الاثنين.

يُبقي ملف parquet على `refund_amount` **عن قصد** — فاليوم الخامس يعيد إنتاج برهان التسريب، ولا
يستطيع ذلك بعمود حُذف هنا. وملف JSON هو ما يمنع ذلك من أن يكون خطرًا: فهو يسجّل أي الأعمدة خصائص،
فيضطر اليوم الخامس إلى اختيار التسريب عمدًا لا مصادفةً.

وهذا الاقتران — البيانات مع العقد الذي يصفها — هو العادة الجديرة بأن تأخذها إلى مشروع تخرّجك.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Write the frame to ARTEFACT_DIR as parquet. Keep the leak column in it.
# 2) Write a JSON sidecar holding the feature list, the target, the categorical
#    columns, the leak column by name, and the prediction moment in words.
# 3) Read both back and print what you got — an artefact you have not reloaded
#    is an artefact you have not tested.
# Search: "pandas to_parquet json dump pathlib"
#
# ١) اكتب الإطار إلى ARTEFACT_DIR بصيغة parquet. وأبقِ عمود التسريب فيه.
# ٢) اكتب ملف JSON مرافقًا يحمل قائمة الخصائص والهدف والأعمدة الفئوية واسم عمود
#    التسريب ولحظة التنبّؤ بالكلمات.
# ٣) أعد قراءة الاثنين واطبع ما حصلت عليه — فالمخرج الذي لم تُعد تحميله مخرج
#    لم تختبره.
# ابحث عن: "pandas to_parquet json dump pathlib"
# ────────────────────────────────────────────────────────────────────

# TODO: Save sales_model_ready.parquet and sales_feature_spec.json, then reload both and print what came back.
# مهمة: احفظ sales_model_ready.parquet وsales_feature_spec.json، ثم أعد تحميلهما واطبع ما عاد منهما.
print(f"saved {table_path.name}: {reloaded.shape[0]:,} rows x {reloaded.shape[1]} columns")
print(f"saved {spec_path.name}: {len(reloaded_spec['features'])} features, "
      f"leak = {reloaded_spec['leak_column']}")
print(f"\nreturn rate in the saved table: {reloaded[TARGET].mean():.1%}")
print(reloaded.dtypes.to_frame('dtype').T.to_string())

## Section 3 — Stretch  (≈30 min)

Open-ended, and lower expectation of completeness.

Add **one** more engineered feature and find out whether it earns its place. Measure the honest AUC
with and without it, using the same split as Task 2.4.

Ideas worth trying: revenue per unit, whether the order was placed at a weekend, the return rate of
the city computed on the *training rows only*, the gap between gross and revenue.

One warning on that third idea, and it is the reason it is listed: a feature built from the target
has to be computed inside the training data only. If you compute a city's return rate over the
whole table and then use it as a column, you have just invented a new leak — a subtler one than
`refund_amount`, and much easier to ship by accident.

**Link to your capstone:** most engineered features do not help. Measuring rather than assuming is
the entire skill, and a feature you tried and rejected is worth writing up in your report.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي (نحو ٣٠ دقيقة)

قسم مفتوح، ولا يُتوقّع إكماله بالكامل.

أضف خاصية مهندسة **واحدة** واكتشف هل تستحق مكانها. قِس المساحة تحت المنحنى النزيهة بوجودها
وبدونها، بالتقسيم نفسه المستخدم في المهمة ٢٫٤.

أفكار تستحق التجربة: الإيراد لكل وحدة، أو هل قُدّم الطلب في عطلة نهاية الأسبوع، أو نسبة إرجاع
المدينة محسوبة على **صفوف التدريب فقط**، أو الفارق بين gross وrevenue.

وتحذير بشأن الفكرة الثالثة، وهو سبب إدراجها: الخاصية المبنية من الهدف يجب أن تُحسب داخل بيانات
التدريب وحدها. فإن حسبت نسبة إرجاع المدينة على الجدول كله ثم استخدمتها عمودًا، تكون قد اخترعت
تسريبًا جديدًا — أدقّ من `refund_amount` وأسهل بكثير في الشحن بالخطأ.

**الصلة بمشروعك:** معظم الخصائص المهندسة لا تفيد. والقياس بدل الافتراض هو المهارة كلها، والخاصية
التي جرّبتها ورفضتها تستحق التوثيق في تقريرك.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# Build your candidate column, add it to a copy of the honest feature list, and
# re-run the same scoring function on the same split. If the AUC does not move,
# say so — a negative result you measured is worth more than a feature you assumed.
#
# ابنِ العمود المرشّح، وأضفه إلى نسخة من قائمة الخصائص النزيهة، وأعد تشغيل دالة
# القياس نفسها على التقسيم نفسه. وإن لم تتحرّك المساحة تحت المنحنى فقل ذلك —
# فالنتيجة السالبة المقيسة أثمن من خاصية مفترضة.
# ────────────────────────────────────────────────────────────────────

# TODO: Add one engineered feature and measure whether it improves the honest AUC.
# مهمة: أضف خاصية مهندسة واحدة وقِس هل تحسّن المساحة تحت المنحنى النزيهة.

**Did it earn its place?** *(one or two sentences — the number, and whether you would keep the
column. "No" is a perfectly good answer.)*

<div dir="rtl" align="right">

**هل استحقّت مكانها؟** *(جملة أو جملتان — الرقم، وهل ستُبقي العمود. و«لا» إجابة جيدة تمامًا.)*

</div>

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخلية أخيرًا. كل فحص يفشل يخبرك بما يجب إصلاحه ولماذا.

</div>

In [ ]:
# --- Sanity checks ----------------------------------------------------------------

check(clean.duplicated().sum() == 0,
      f"no duplicate rows should remain, found {int(clean.duplicated().sum())}",
      f"يجب ألا تبقى صفوف مكرّرة، ووُجد {int(clean.duplicated().sum())}")

check(clean["city"].nunique() == 5,
      f"city should collapse to 5 cities, got {clean['city'].nunique()}: "
      f"{sorted(clean['city'].unique())}",
      f"يجب أن يتجمّع city في خمس مدن، والناتج {clean['city'].nunique()}: "
      f"{sorted(clean['city'].unique())}")

check(n_unparsed == 0 and n_future == 0 and n_before_order == 0,
      f"dates should parse cleanly: {n_unparsed} unparsed, {n_future} future, "
      f"{n_before_order} shipped before ordered",
      f"يجب أن تُحلّل التواريخ بنظافة: {n_unparsed} غير محلَّل، {n_future} مستقبلي، "
      f"{n_before_order} شُحن قبل الطلب")

check(clean["discount_pct"].isna().sum() == 0 and clean["discount_missing"].mean() > 0.10,
      f"discount_pct should be fully imputed with the missingness kept: "
      f"{int(clean['discount_pct'].isna().sum())} still missing, indicator "
      f"{clean['discount_missing'].mean():.1%}",
      f"يجب تعويض discount_pct بالكامل مع الإبقاء على الغياب: "
      f"{int(clean['discount_pct'].isna().sum())} ما زال مفقودًا، والمؤشّر "
      f"{clean['discount_missing'].mean():.1%}")

check(LEAK not in FEATURES,
      f"{LEAK} must not be in the feature list — it is leakage",
      f"يجب ألا يكون {LEAK} في قائمة الخصائص — فهو تسريب")

check(leaked_auc - honest_auc > 0.20,
      f"the leak should be dramatic: honest {honest_auc:.4f} vs leaked {leaked_auc:.4f}",
      f"يجب أن يكون التسريب صارخًا: النزيه {honest_auc:.4f} مقابل المُسرَّب {leaked_auc:.4f}")

check(not unaccounted and not overlap,
      f"every column must be classified exactly once — unaccounted: {sorted(unaccounted)}, "
      f"in two lists: {sorted(overlap)}",
      f"يجب تصنيف كل عمود مرة واحدة بالضبط — غير محسوب: {sorted(unaccounted)}، "
      f"في قائمتين: {sorted(overlap)}")

check((ARTEFACT_DIR / "sales_model_ready.parquet").exists()
      and (ARTEFACT_DIR / "sales_feature_spec.json").exists(),
      "both sales_model_ready.parquet and sales_feature_spec.json should exist — D5 loads them",
      "يجب أن يوجد الملفان sales_model_ready.parquet وsales_feature_spec.json — يحمّلهما اليوم الخامس")

report()

## What's next

D4 is a **holiday** — there is no lab.

D5 picks up exactly here. It loads `sales_model_ready.parquet` and `sales_feature_spec.json`, trains
a classifier on the features you approved, and discovers that a model can be 77% accurate while
being nearly useless. Then it puts `refund_amount` back in, watches the AUC jump to about 0.999,
and explains why that number is worthless.

<div dir="rtl" align="right">

## ماذا بعد

اليوم الرابع **عطلة** — لا معمل فيه.

ويلتقط اليوم الخامس الخيط من هنا تمامًا. فيحمّل `sales_model_ready.parquet`
و`sales_feature_spec.json`، ويدرّب مصنّفًا على الخصائص التي اعتمدتها، ويكتشف أن النموذج قد يبلغ
دقّة ٧٧٪ وهو شبه عديم الفائدة. ثم يعيد `refund_amount` ويشاهد المساحة تحت المنحنى تقفز إلى نحو
٠٫٩٩٩، ويشرح لماذا هذا الرقم بلا قيمة.

</div>